# Environment Validation

Before implementing the distributed profiling pipeline, the development environment is validated to ensure that all required system, C++20, networking, database, debugging, and GPU components are available and working correctly.

The environment validation covers the following areas.

## 1. System and GPU Environment

Verify the underlying Linux environment and GPU runtime.

Checked components:

- Linux kernel and OS version
- NVIDIA GPU availability
- NVIDIA Tesla T4 GPU
- NVIDIA driver
- CUDA runtime
- CUDA compiler (`nvcc`)

This ensures that the environment can support both the host-side C++ infrastructure and GPU workloads used later for profiling experiments.

---

## 2. Modern C++20 Support

Verify that the installed C++ compiler supports the C++20 features required by the project.

Tested features include:

- `std::span`
- `std::variant`
- `std::optional`
- C++20 ranges
- standard C++20 compilation flags

The project relies on modern C++ abstractions to build modular event representations, APIs, and data-processing components.

---

## 3. C++20 Thread Management

Verify support for modern thread lifecycle management.

Tested features include:

- `std::thread`
- `std::jthread`
- `std::stop_token`
- cooperative thread cancellation
- automatic thread joining

These features will later be used to implement asynchronous workers and graceful shutdown of the profiling pipeline.

---

## 4. Multithreading and Synchronization

Verify that standard synchronization primitives work correctly.

Tested components include:

- `std::mutex`
- `std::lock_guard`
- `std::unique_lock`
- `std::condition_variable`
- producer-consumer synchronization
- inter-thread notification

These primitives form the foundation of the bounded event queue used between profiling-event producers and asynchronous consumers.

---

## 5. TCP Client-Server Communication

Verify that POSIX TCP sockets can be used inside the environment.

The test includes:

- TCP socket creation
- server binding
- listening for incoming connections
- accepting a client connection
- establishing a localhost TCP connection
- sending data from a client
- receiving data on a server
- closing socket resources correctly

This validates the basic transport layer required for profiling agents to transfer events to a centralized collector.

---

## 6. SQLite Database Support

Verify that profiling data can be stored using SQLite.

The test includes:

- opening a database
- creating a table
- executing SQL commands
- closing the database safely
- verifying the resulting database schema

SQLite will later be used as a lightweight storage backend for collected profiling events.

---

## 7. CMake and Build System

Verify that the project can be built using a production-style C++ build workflow.

Checked tools include:

- CMake
- Ninja
- C++20 project configuration
- out-of-source builds
- executable generation

This ensures that the project can be organized as a modular C++ codebase rather than a collection of standalone source files.

---

## 8. AddressSanitizer

Verify that AddressSanitizer can detect memory-safety errors.

The validation intentionally introduces an out-of-bounds memory access and checks whether the sanitizer reports the error.

This will later help detect issues such as:

- buffer overflows
- use-after-free errors
- invalid memory accesses
- memory-lifetime bugs

---

## 9. ThreadSanitizer

Verify that ThreadSanitizer can detect concurrency errors.

The validation intentionally introduces a data race between multiple threads.

ThreadSanitizer will later be useful for detecting:

- data races
- unsafe shared-memory access
- synchronization errors

Because multithreading is a major part of the profiling pipeline, concurrency validation is especially important.

---

## 10. CUDA C++ Execution

Verify that a native CUDA C++ program can be compiled and executed on the Tesla T4 GPU.

The test includes:

- GPU memory allocation
- host-to-device memory transfer
- CUDA kernel launch
- GPU synchronization
- device-to-host memory transfer
- result validation

A simple vector-addition kernel is used as the initial GPU workload.

This confirms that the environment can later generate real GPU activity for profiling experiments.

---

# Environment Validation Goal

After completing these tests, the environment should support the complete initial development stack:

```text
Linux
  +
C++20
  +
Multithreading
  +
POSIX TCP Networking
  +
SQLite
  +
CMake / Ninja
  +
ASan / TSan
  +
CUDA / Tesla T4
```

Once these components are validated, the project can move on to implementing the first end-to-end profiling data path:

```text
GPU / Synthetic Event Source
            ↓
       ProfileEvent
            ↓
       BoundedQueue
            ↓
      Async Consumer
            ↓
        TCP Client
            ↓
      Collector Server
            ↓
          Storage
```

The purpose of this environment validation is to isolate infrastructure issues early, so later development can focus on system design, concurrency, networking, scalability, and profiling-data processing rather than toolchain setup.

### system / GPU / CUDA

In [1]:
!uname -a
!cat /etc/os-release

!nvidia-smi
!nvcc --version

Linux 77b156ac78bd 6.6.122+ #1 SMP Thu Apr 30 18:17:14 UTC 2026 x86_64 x86_64 x86_64 GNU/Linux
PRETTY_NAME="Ubuntu 24.04.4 LTS"
NAME="Ubuntu"
VERSION_ID="24.04"
VERSION="24.04.4 LTS (Noble Numbat)"
VERSION_CODENAME=noble
ID=ubuntu
ID_LIKE=debian
HOME_URL="https://www.ubuntu.com/"
SUPPORT_URL="https://help.ubuntu.com/"
BUG_REPORT_URL="https://bugs.launchpad.net/ubuntu/"
PRIVACY_POLICY_URL="https://www.ubuntu.com/legal/terms-and-policies/privacy-policy"
UBUNTU_CODENAME=noble
LOGO=ubuntu-logo
Sat Oct  3 12:41:52 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|            

### C++ compiler

In [2]:
!g++ --version
!clang++ --version

g++ (Ubuntu 13.3.0-6ubuntu2~24.04.1) 13.3.0
Copyright (C) 2023 Free Software Foundation, Inc.
This is free software; see the source for copying conditions.  There is NO
warranty; not even for MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE.

/bin/bash: line 1: clang++: command not found


In [3]:
%%writefile test_cpp20.cpp

#include <iostream>
#include <vector>
#include <span>
#include <variant>
#include <optional>
#include <ranges>

int main() {
    std::vector<int> data{1, 2, 3, 4, 5};

    std::span<int> view(data);

    std::variant<int, std::string> value = std::string("C++20 works");

    std::optional<int> result = 42;

    std::cout << std::get<std::string>(value) << "\n";
    std::cout << "span size = " << view.size() << "\n";
    std::cout << "optional = " << *result << "\n";

    for (int x : data | std::views::filter([](int x) {
        return x % 2 == 0;
    })) {
        std::cout << x << " ";
    }

    std::cout << "\n";
}

Writing test_cpp20.cpp


In [4]:
!g++ -std=c++20 -Wall -Wextra -Wpedantic test_cpp20.cpp -o test_cpp20
!./test_cpp20

C++20 works
span size = 5
optional = 42
2 4 


### std::jthread and stop_token

In [5]:
%%writefile test_jthread.cpp

#include <chrono>
#include <iostream>
#include <thread>

using namespace std::chrono_literals;

void worker(std::stop_token stop_token) {
    int count = 0;

    while (!stop_token.stop_requested()) {
        std::cout << "worker iteration: " << count++ << "\n";
        std::this_thread::sleep_for(100ms);
    }

    std::cout << "worker received stop request\n";
}

int main() {
    std::jthread thread(worker);

    std::this_thread::sleep_for(500ms);

    thread.request_stop();

    return 0;
}

Writing test_jthread.cpp


In [6]:
!g++ -std=c++20 -pthread test_jthread.cpp -o test_jthread
!./test_jthread

worker iteration: 0
worker iteration: 1
worker iteration: 2
worker iteration: 3
worker iteration: 4
worker received stop request


### mutex + condition_variable

In [7]:
%%writefile test_threading.cpp

#include <condition_variable>
#include <iostream>
#include <mutex>
#include <queue>
#include <thread>

std::queue<int> queue;
std::mutex mutex;
std::condition_variable cv;

bool done = false;

void producer() {
    for (int i = 0; i < 5; ++i) {
        {
            std::lock_guard<std::mutex> lock(mutex);
            queue.push(i);
        }

        cv.notify_one();
    }

    {
        std::lock_guard<std::mutex> lock(mutex);
        done = true;
    }

    cv.notify_all();
}

void consumer() {
    while (true) {
        std::unique_lock<std::mutex> lock(mutex);

        cv.wait(lock, [] {
            return !queue.empty() || done;
        });

        while (!queue.empty()) {
            int value = queue.front();
            queue.pop();

            lock.unlock();

            std::cout << "Consumed: " << value << "\n";

            lock.lock();
        }

        if (done) {
            break;
        }
    }
}

int main() {
    std::thread producer_thread(producer);
    std::thread consumer_thread(consumer);

    producer_thread.join();
    consumer_thread.join();

    return 0;
}

Writing test_threading.cpp


In [8]:
!g++ -std=c++20 -pthread test_threading.cpp -o test_threading
!./test_threading

Consumed: 0
Consumed: 1
Consumed: 2
Consumed: 3
Consumed: 4


### TCP socket

In [9]:
%%writefile server.cpp

#include <arpa/inet.h>
#include <cstring>
#include <iostream>
#include <sys/socket.h>
#include <unistd.h>

int main() {
    int server_fd = socket(AF_INET, SOCK_STREAM, 0);

    if (server_fd < 0) {
        std::cerr << "socket creation failed\n";
        return 1;
    }

    int opt = 1;
    setsockopt(
        server_fd,
        SOL_SOCKET,
        SO_REUSEADDR,
        &opt,
        sizeof(opt)
    );

    sockaddr_in address{};
    address.sin_family = AF_INET;
    address.sin_addr.s_addr = INADDR_ANY;
    address.sin_port = htons(9000);

    if (bind(
            server_fd,
            reinterpret_cast<sockaddr*>(&address),
            sizeof(address)
        ) < 0) {
        std::cerr << "bind failed\n";
        return 1;
    }

    if (listen(server_fd, 4) < 0) {
        std::cerr << "listen failed\n";
        return 1;
    }

    std::cout << "Server listening on port 9000\n";

    sockaddr_in client_address{};
    socklen_t client_len = sizeof(client_address);

    int client_fd = accept(
        server_fd,
        reinterpret_cast<sockaddr*>(&client_address),
        &client_len
    );

    char buffer[1024]{};

    ssize_t bytes = recv(
        client_fd,
        buffer,
        sizeof(buffer) - 1,
        0
    );

    if (bytes > 0) {
        std::cout
            << "Received: "
            << buffer
            << "\n";
    }

    close(client_fd);
    close(server_fd);

    return 0;
}

Writing server.cpp


In [10]:
%%writefile client.cpp

#include <arpa/inet.h>
#include <cstring>
#include <iostream>
#include <sys/socket.h>
#include <unistd.h>

int main() {
    int socket_fd = socket(AF_INET, SOCK_STREAM, 0);

    sockaddr_in server_address{};
    server_address.sin_family = AF_INET;
    server_address.sin_port = htons(9000);

    inet_pton(
        AF_INET,
        "127.0.0.1",
        &server_address.sin_addr
    );

    if (connect(
            socket_fd,
            reinterpret_cast<sockaddr*>(&server_address),
            sizeof(server_address)
        ) < 0) {
        std::cerr << "connect failed\n";
        return 1;
    }

    const char* message =
        "Hello from profiler agent";

    send(
        socket_fd,
        message,
        std::strlen(message),
        0
    );

    close(socket_fd);

    return 0;
}

Writing client.cpp


In [11]:
!g++ -std=c++20 server.cpp -o server
!g++ -std=c++20 client.cpp -o client

In [13]:
!./server > server.log 2>&1 &

In [16]:
!sleep 1
!./client
!sleep 1
!cat server.log

connect failed
bind failed


In [38]:
!ps aux | grep server
!ss -ltnp | grep 9000

root         124  0.4  1.2 629184 163664 ?       Sl   12:33   0:07 /usr/bin/python3 /usr/local/bin/jupyter-server --debug --transport="ipc" --ip=172.28.0.12 --ServerApp.token= --port=9000 --FileContentsManager.root_dir=/ --FileContentsManager.allow_hidden=True --ServerApp.log_format="|%(levelname)s|%(message)s" --ServerApp.iopub_data_rate_limit=1e10 --MappingKernelManager.root_dir=/content
root        2162  0.0  0.1 1275484 21688 ?       Sl   12:41   0:00 /usr/colab/bin/language_service -- node /datalab/web/pyright/pyright-langserver.js --stdio --cancellationReceive=file:a7c7b390c8ee42d8797deae3e1da9afe5601d06eb8
root        2173  0.8  1.2 1671224 165364 ?      Sl   12:41   0:09 node /datalab/web/pyright/pyright-langserver.js --stdio --cancellationReceive=file:a7c7b390c8ee42d8797deae3e1da9afe5601d06eb8
root        8492  0.0  0.0   7348  3680 ?        S    13:00   0:00 /bin/bash -c ps aux | grep server
root        8494  0.0  0.0   6552  2512 ?        S    13:00   0:00 grep server
LISTEN

In [39]:
!pkill -f "./server"

In [40]:
!ss -ltnp | grep 9000

LISTEN 0      128      172.28.0.12:9000       0.0.0.0:*    users:(("jupyter-server",pid=124,fd=7))


In [41]:
%%writefile server.cpp

#include <arpa/inet.h>
#include <cerrno>
#include <cstring>
#include <iostream>
#include <sys/socket.h>
#include <unistd.h>

int main() {
    int server_fd = socket(AF_INET, SOCK_STREAM, 0);

    if (server_fd < 0) {
        std::cerr
            << "socket() failed: "
            << std::strerror(errno)
            << "\n";
        return 1;
    }

    int opt = 1;

    if (setsockopt(
            server_fd,
            SOL_SOCKET,
            SO_REUSEADDR,
            &opt,
            sizeof(opt)
        ) < 0) {

        std::cerr
            << "setsockopt() failed: "
            << std::strerror(errno)
            << "\n";

        close(server_fd);
        return 1;
    }

    sockaddr_in address{};
    address.sin_family = AF_INET;
    address.sin_addr.s_addr = htonl(INADDR_LOOPBACK);
    address.sin_port = htons(19000);

    if (bind(
            server_fd,
            reinterpret_cast<sockaddr*>(&address),
            sizeof(address)
        ) < 0) {

        std::cerr
            << "bind() failed: "
            << std::strerror(errno)
            << "\n";

        close(server_fd);
        return 1;
    }

    if (listen(server_fd, 4) < 0) {

        std::cerr
            << "listen() failed: "
            << std::strerror(errno)
            << "\n";

        close(server_fd);
        return 1;
    }

    std::cout
        << "Server listening on 127.0.0.1:19000"
        << std::endl;

    sockaddr_in client_address{};
    socklen_t client_len = sizeof(client_address);

    int client_fd = accept(
        server_fd,
        reinterpret_cast<sockaddr*>(&client_address),
        &client_len
    );

    if (client_fd < 0) {
        std::cerr
            << "accept() failed: "
            << std::strerror(errno)
            << "\n";

        close(server_fd);
        return 1;
    }

    char buffer[1024]{};

    ssize_t bytes = recv(
        client_fd,
        buffer,
        sizeof(buffer) - 1,
        0
    );

    if (bytes < 0) {
        std::cerr
            << "recv() failed: "
            << std::strerror(errno)
            << "\n";
    }
    else {
        buffer[bytes] = '\0';

        std::cout
            << "Received: "
            << buffer
            << std::endl;
    }

    close(client_fd);
    close(server_fd);

    return 0;
}

Overwriting server.cpp


In [42]:
%%writefile client.cpp

#include <arpa/inet.h>
#include <cerrno>
#include <cstring>
#include <iostream>
#include <sys/socket.h>
#include <unistd.h>

int main() {
    int socket_fd = socket(AF_INET, SOCK_STREAM, 0);

    if (socket_fd < 0) {
        std::cerr
            << "socket() failed: "
            << std::strerror(errno)
            << "\n";
        return 1;
    }

    sockaddr_in server_address{};
    server_address.sin_family = AF_INET;
    server_address.sin_port = htons(19000);

    if (inet_pton(
            AF_INET,
            "127.0.0.1",
            &server_address.sin_addr
        ) <= 0) {

        std::cerr << "inet_pton() failed\n";

        close(socket_fd);
        return 1;
    }

    if (connect(
            socket_fd,
            reinterpret_cast<sockaddr*>(&server_address),
            sizeof(server_address)
        ) < 0) {

        std::cerr
            << "connect() failed: "
            << std::strerror(errno)
            << "\n";

        close(socket_fd);
        return 1;
    }

    const char* message =
        "Hello from profiler agent";

    ssize_t sent = send(
        socket_fd,
        message,
        std::strlen(message),
        0
    );

    if (sent < 0) {
        std::cerr
            << "send() failed: "
            << std::strerror(errno)
            << "\n";
    }

    close(socket_fd);

    return 0;
}

Overwriting client.cpp


In [43]:
!g++ -std=c++20 server.cpp -o server
!g++ -std=c++20 client.cpp -o client

In [44]:
!./server > server.log 2>&1 &
!sleep 2
!cat server.log
!ss -ltnp | grep 19000

Server listening on 127.0.0.1:19000
LISTEN 0      4          127.0.0.1:19000      0.0.0.0:*    users:(("server",pid=8952,fd=3))       


In [45]:
!./client
!sleep 1
!cat server.log

Server listening on 127.0.0.1:19000
Received: Hello from profiler agent


### SQLite

In [18]:
!sqlite3 --version

/bin/bash: line 1: sqlite3: command not found


In [19]:
!sudo apt-get update -qq
!sudo apt-get install -y sqlite3 libsqlite3-dev

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
libsqlite3-dev is already the newest version (3.45.1-1ubuntu2.8).
libsqlite3-dev set to manually installed.
Suggested packages:
  sqlite3-doc
The following NEW packages will be installed:
  sqlite3
0 upgraded, 1 newly installed, 0 to remove and 47 not upgraded.
Need to get 144 kB of archives.
After this operation, 584 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu noble-updates/main amd64 sqlite3 amd64 3.45.1-1ubuntu2.8 [144 kB]
Fetched 144 kB in 1s (108 kB/s)
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 79, <STDIN> line 

In [20]:
%%writefile test_sqlite.cpp

#include <sqlite3.h>
#include <iostream>

int main() {
    sqlite3* db = nullptr;

    if (sqlite3_open(
            "profiling.db",
            &db
        ) != SQLITE_OK) {
        std::cerr << "Failed to open database\n";
        return 1;
    }

    const char* sql = R"(
        CREATE TABLE IF NOT EXISTS events (
            id INTEGER PRIMARY KEY,
            kernel_name TEXT,
            duration_ns INTEGER
        );
    )";

    char* error_message = nullptr;

    if (sqlite3_exec(
            db,
            sql,
            nullptr,
            nullptr,
            &error_message
        ) != SQLITE_OK) {

        std::cerr
            << "SQLite error: "
            << error_message
            << "\n";

        sqlite3_free(error_message);
    }

    sqlite3_close(db);

    std::cout << "SQLite test passed\n";

    return 0;
}

Writing test_sqlite.cpp


In [21]:
!g++ -std=c++20 test_sqlite.cpp -lsqlite3 -o test_sqlite
!./test_sqlite

SQLite test passed


In [22]:
!sqlite3 profiling.db ".tables"

events


### CMake

In [23]:
!cmake --version
!ninja --version

cmake version 3.31.10

CMake suite maintained and supported by Kitware (kitware.com/cmake).
/bin/bash: line 1: ninja: command not found


In [24]:
!sudo apt-get install -y ninja-build

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  ninja-build
0 upgraded, 1 newly installed, 0 to remove and 47 not upgraded.
Need to get 129 kB of archives.
After this operation, 364 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu noble/universe amd64 ninja-build amd64 1.11.1-2 [129 kB]
Fetched 129 kB in 1s (116 kB/s)
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 79, <STDIN> line 1.)
debconf: falling back to frontend: Readline
debconf: unable to initialize frontend: Readline
debconf: (This frontend requires a controlling tty.)
debconf: falling back to frontend: Teletype
dpkg-preconfigure: unable to re-open stdin: 
Selecting previously unselected package ninja-build.
(Reading database ... 127338 files and direc

In [25]:
!mkdir -p cpp_test/src

In [26]:
%%writefile cpp_test/src/main.cpp

#include <iostream>

int main() {
    std::cout << "CMake C++20 environment ready\n";
}

Writing cpp_test/src/main.cpp


In [27]:
%%writefile cpp_test/CMakeLists.txt

cmake_minimum_required(VERSION 3.16)

project(
    distributed_profiler_test
    LANGUAGES CXX
)

set(CMAKE_CXX_STANDARD 20)
set(CMAKE_CXX_STANDARD_REQUIRED ON)

add_executable(
    environment_test
    src/main.cpp
)

Writing cpp_test/CMakeLists.txt


In [28]:
!cmake -S cpp_test -B cpp_test/build -G Ninja
!cmake --build cpp_test/build
!./cpp_test/build/environment_test

-- The CXX compiler identification is GNU 13.3.0
-- Detecting CXX compiler ABI info
-- Detecting CXX compiler ABI info - done
-- Check for working CXX compiler: /usr/bin/c++ - skipped
-- Detecting CXX compile features
-- Detecting CXX compile features - done
-- Configuring done (0.3s)
-- Generating done (0.0s)
-- Build files have been written to: /content/cpp_test/build
[2/2] Linking CXX executable environment_test
CMake C++20 environment ready


### AddressSanitizer

In [29]:
%%writefile test_asan.cpp

#include <iostream>

int main() {
    int* data = new int[4];

    data[5] = 42;

    delete[] data;

    return 0;
}

Writing test_asan.cpp


In [30]:
!g++ \
    -std=c++20 \
    -fsanitize=address \
    -fno-omit-frame-pointer \
    test_asan.cpp \
    -o test_asan

In [31]:
!./test_asan

==7746==ERROR: AddressSanitizer: heap-buffer-overflow on address 0x502000000024 at pc 0x587886dba262 bp 0x7ffd4bdb25a0 sp 0x7ffd4bdb2590
WRITE of size 4 at 0x502000000024 thread T0
    #0 0x587886dba261 in main (/content/test_asan+0x1261) (BuildId: 1adb39e63fc2500d735824eceaaeff7748625fbf)
    #1 0x7f19862b21c9  (/lib/x86_64-linux-gnu/libc.so.6+0x2a1c9) (BuildId: a4a7992a8e66555c8141ab2a08a8465ff6e0ea65)
    #2 0x7f19862b228a in __libc_start_main (/lib/x86_64-linux-gnu/libc.so.6+0x2a28a) (BuildId: a4a7992a8e66555c8141ab2a08a8465ff6e0ea65)
    #3 0x587886dba144 in _start (/content/test_asan+0x1144) (BuildId: 1adb39e63fc2500d735824eceaaeff7748625fbf)

0x502000000024 is located 4 bytes after 16-byte region [0x502000000010,0x502000000020)
allocated by thread T0 here:
    #0 0x7f198692e6c8 in operator new[](unsigned long) ../../../../src/libsanitizer/asan/asan_new_delete.cpp:98
    #1 0x587886dba21e in main (/content/test_asan+0x121e) (BuildId: 1adb39e63fc2500d735824eceaaeff7748625fbf)
    

### ThreadSanitizer

In [32]:
%%writefile test_tsan.cpp

#include <iostream>
#include <thread>

int counter = 0;

void increment() {
    for (int i = 0; i < 100000; ++i) {
        ++counter;
    }
}

int main() {
    std::thread t1(increment);
    std::thread t2(increment);

    t1.join();
    t2.join();

    std::cout << counter << "\n";

    return 0;
}

Writing test_tsan.cpp


In [33]:
!g++ \
    -std=c++20 \
    -pthread \
    -fsanitize=thread \
    test_tsan.cpp \
    -o test_tsan

In [35]:
!./test_tsan

  Read of size 4 at 0x57d9e7025154 by thread T2:
    #0 increment() <null> (test_tsan+0x1399) (BuildId: af48cbb2527d8ad13667dfe1da5c0c0344b80a78)
    #1 void std::__invoke_impl<void, void (*)()>(std::__invoke_other, void (*&&)()) <null> (test_tsan+0x2196) (BuildId: af48cbb2527d8ad13667dfe1da5c0c0344b80a78)
    #2 std::__invoke_result<void (*)()>::type std::__invoke<void (*)()>(void (*&&)()) <null> (test_tsan+0x20eb) (BuildId: af48cbb2527d8ad13667dfe1da5c0c0344b80a78)
    #3 void std::thread::_Invoker<std::tuple<void (*)()> >::_M_invoke<0ul>(std::_Index_tuple<0ul>) <null> (test_tsan+0x2040) (BuildId: af48cbb2527d8ad13667dfe1da5c0c0344b80a78)
    #4 std::thread::_Invoker<std::tuple<void (*)()> >::operator()() <null> (test_tsan+0x1fe2) (BuildId: af48cbb2527d8ad13667dfe1da5c0c0344b80a78)
    #5 std::thread::_State_impl<std::thread::_Invoker<std::tuple<void (*)()> > >::_M_run() <null> (test_tsan+0x1f94) (BuildId: af48cbb2527d8ad13667dfe1da5c0c0344b80a78)
    #6 <null> <null> (libstdc++.so.6

### CUDA C++

In [36]:
%%writefile test_cuda.cu

#include <cuda_runtime.h>

#include <iostream>

__global__
void vector_add(
    const float* a,
    const float* b,
    float* c,
    int n
) {
    int index =
        blockIdx.x * blockDim.x +
        threadIdx.x;

    if (index < n) {
        c[index] =
            a[index] +
            b[index];
    }
}

int main() {
    constexpr int N = 1 << 20;

    size_t bytes =
        N * sizeof(float);

    float* h_a = new float[N];
    float* h_b = new float[N];
    float* h_c = new float[N];

    for (int i = 0; i < N; ++i) {
        h_a[i] = 1.0f;
        h_b[i] = 2.0f;
    }

    float* d_a;
    float* d_b;
    float* d_c;

    cudaMalloc(&d_a, bytes);
    cudaMalloc(&d_b, bytes);
    cudaMalloc(&d_c, bytes);

    cudaMemcpy(
        d_a,
        h_a,
        bytes,
        cudaMemcpyHostToDevice
    );

    cudaMemcpy(
        d_b,
        h_b,
        bytes,
        cudaMemcpyHostToDevice
    );

    int block_size = 256;

    int grid_size =
        (N + block_size - 1)
        / block_size;

    vector_add<<<
        grid_size,
        block_size
    >>>(
        d_a,
        d_b,
        d_c,
        N
    );

    cudaDeviceSynchronize();

    cudaMemcpy(
        h_c,
        d_c,
        bytes,
        cudaMemcpyDeviceToHost
    );

    std::cout
        << "Result: "
        << h_c[0]
        << "\n";

    cudaFree(d_a);
    cudaFree(d_b);
    cudaFree(d_c);

    delete[] h_a;
    delete[] h_b;
    delete[] h_c;

    return 0;
}

Writing test_cuda.cu


In [37]:
!nvcc -std=c++17 test_cuda.cu -o test_cuda
!./test_cuda

Result: 3
